In [ ]:
# ============================================================================
# CÉLULA COMPLEMENTAR PARA: notebooks/01_eda_estatistica.ipynb
# DESCRIÇÃO: Auditoria de Valores Ausentes (NaN), Duplicados e Outliers via Z-Score
# ============================================================================
import pyspark.sql.functions as F
import pandas as pd

def auditoria_qualidade_e_zscore(spark_session, nome_tabela: str, coluna_metrica: str):
    """
    Documenta a contagem de valores nulos/NaN, registos duplicados
    e deteta outliers matemáticos via Z-Score (|Z| > 3).
    """
    df = spark_session.table(nome_tabela)
    total_linhas = df.count()
    linhas_distintas = df.distinct().count()
    duplicados = total_linhas - linhas_distintas

    # 1. Levantamento de Nulos / NaNs por coluna
    exprs_nulos = [
        F.sum(F.when(F.col(c).isNull() | F.isnan(c) if dict(df.dtypes)[c] in ('double', 'float') else F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    df_nulos = df.agg(*exprs_nulos).toPandas().T.reset_index()
    df_nulos.columns = ["Coluna", "Qtd_Nulos"]
    df_nulos["Percentual_Nulos (%)"] = ((df_nulos["Qtd_Nulos"] / total_linhas) * 100).round(2)

    # 2. Cálculo de Outliers via Z-Score (|Z| > 3 desvios padrão da média)
    stats = df.select(
        F.mean(F.col(coluna_metrica)).alias("media"),
        F.stddev(F.col(coluna_metrica)).alias("desvio")
    ).collect()[0]

    media_val, desvio_val = stats["media"], stats["desvio"]
    df_zscore = df.withColumn(
        "z_score",
        F.round((F.col(coluna_metrica) - F.lit(media_val)) / F.lit(desvio_val), 2)
    ).withColumn(
        "is_outlier_zscore",
        F.when(F.abs(F.col("z_score")) > 3.0, True).otherwise(False)
    )
    qtd_outliers_z = df_zscore.filter(F.col("is_outlier_zscore") == True).count()

    print(f"=== RAIO-X DE QUALIDADE: {nome_tabela} ===")
    print(f"Total de Registos: {total_linhas} | Duplicados Exatos: {duplicados}")
    print(f"Média ({coluna_metrica}): {media_val:.2f} | Desvio Padrão: {desvio_val:.2f}")
    print(f"Outliers pelo critério Z-Score (|Z| > 3): {qtd_outliers_z} registos\n")

    return df_nulos, df_zscore.filter(F.col("is_outlier_zscore") == True)

# Execução sobre a tabela de empréstimos e transações da camada Gold
tabela_nulos, df_outliers_z = auditoria_qualidade_e_zscore(
    spark, "portfolio.gold.fato_emprestimos", "valor_emprestimo"
)
display(spark.createDataFrame(tabela_nulos))
display(df_outliers_z)

In [0]:
display(df_spark.dtypes)

_1,_2
id_emprestimo,string
id_conta,string
data_emprestimo,date
valor_emprestimo,int
duracao,int
parcelas,double
status_emprestimo,string


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType
import pandas as pd

# 1. Lendo a tabela
df_spark = spark.table("portfolio.gold.fato_transacoes")

def descritivas_profissional(df):
    # Identifica as colunas numéricas de forma nativa e estável
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    
    if not numeric_cols:
        return "Nenhuma coluna numérica foi encontrada!"
    
    # 1. Executa o resumo nativo do Spark
    summary_df = df.select(*numeric_cols).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
    
    # 2. O PULO DO GATO (CORRIGIDO): toPandas() com 'P' maiúsculo e sem underline
    desc_pd = summary_df.toPandas().set_index("summary").T
    desc_pd = desc_pd.rename(columns={"stddev": "std"})
    
    # Converte as métricas para formato numérico no Pandas local para podermos calcular o CV
    for col in desc_pd.columns:
        desc_pd[col] = pd.to_numeric(desc_pd[col], errors='coerce')
    
    # 3. Calcula Skewness e Kurtosis nativamente no Spark em um único Job rápido
    extra_exprs = []
    for c in numeric_cols:
        extra_exprs.append(F.skewness(c).alias(f"{c}_Skew"))
        extra_exprs.append(F.kurtosis(c).alias(f"{c}_Kurtosis"))
        
    extra_res = df.agg(*extra_exprs).toPandas()
    
    # Mapeia com total segurança usando os nomes reais das colunas que estão no Index
    desc_pd["Skew"] = [extra_res[f"{c}_Skew"][0] for c in desc_pd.index]
    desc_pd["Kurtosis"] = [extra_res[f"{c}_Kurtosis"][0] for c in desc_pd.index]
    
    # Calcula o Coeficiente de Variação
    desc_pd["CV"] = desc_pd["std"] / desc_pd["mean"]
    
    # Ordenação final das colunas
    ordered_cols = [
        "count", "mean", "std", "CV",
        "min", "25%", "50%", "75%", "max",
        "Skew", "Kurtosis"
    ]
    desc_pd = desc_pd[ordered_cols].round(2)
    
    # Transforma o Index (que agora tem os nomes certos) em uma coluna real para o Databricks exibir
    desc_pd = desc_pd.reset_index().rename(columns={"index": "Nome_da_Variavel"})
    
    return desc_pd

# Executando
resultado = descritivas_profissional(df_spark)

# Exibindo no Databricks
if isinstance(resultado, str):
    print(resultado)
else:
    # Converte de volta para Spark DataFrame para uma renderização perfeita e sem avisos na tela
    df_final_spark = spark.createDataFrame(resultado)
    display(df_final_spark)

Nome_da_Variavel,count,mean,std,CV,min,25%,50%,75%,max,Skew,Kurtosis
id_transacao,1056320,1335310.7,1227486.51,0.92,1.0,430247.0,858537.0,2060972.0,3682987.0,0.95,-0.66
id_conta,1056320,2936.87,2477.35,0.84,1.0,1205.0,2434.0,3660.0,11382.0,1.53,2.1
valor,1056320,5924.15,9522.74,1.61,0.0,136.0,2100.0,6800.0,87400.0,2.59,7.76
saldo,1056320,38518.33,22117.87,0.57,-41125.7,22400.0,33141.6,49596.2,209637.0,1.23,1.88


#### Outliers

In [0]:
import pyspark.sql.functions as F

def flag_outliers(df, coluna_alvo):
    # Calcula Q1 e Q3 de forma eficiente
    quantiles = df.approxQuantile(coluna_alvo, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    
    # Cria uma nova coluna sinalizando se a linha é um outlier
    df_resultado = df.withColumn(
        "is_outlier",
        F.when((F.col(coluna_alvo) < limite_inferior) | (F.col(coluna_alvo) > limite_superior), True).otherwise(False)
    )
    
    print(f"Limites para {coluna_alvo} -> Inferior: {limite_inferior} | Superior: {limite_superior}\n")
    return df_resultado

# ---------------------------------------------------------
# AJUSTADO: Agora usando a coluna real 'valor_emprestimo'
df_analisado = flag_outliers(df_spark, "valor_emprestimo")

# Exibe apenas as linhas que são outliers para você inspecionar o que está fora do padrão
display(df_analisado.filter(F.col("is_outlier") == True))

Limites para valor_emprestimo -> Inferior: -146400.0 | Superior: 421152.0



id_emprestimo,id_conta,data_emprestimo,valor_emprestimo,duracao,parcelas,status_emprestimo,is_outlier
6228,6034,1993-12-01,464520,60,7742.0,Encerrado - Inadimplente,true
7142,10451,1994-12-19,482940,60,8049.0,Ativo - Inadimplente,true
5132,817,1995-02-17,538500,60,8975.0,Ativo - Regular,true
5970,4774,1995-07-09,465072,48,9689.0,Ativo - Regular,true
5725,3679,1996-07-23,444864,48,9268.0,Ativo - Regular,true
6415,6950,1997-02-12,475680,48,9910.0,Ativo - Regular,true
5486,2516,1997-06-13,466608,48,9721.0,Ativo - Regular,true
6625,7966,1997-09-07,473280,60,7888.0,Ativo - Inadimplente,true
6534,7542,1997-10-19,590820,60,9847.0,Ativo - Regular,true
5731,3711,1997-11-05,460980,60,7683.0,Ativo - Inadimplente,true


In [0]:
%sql
select* from portfolio.gold.dim_cliente


id_conta,id_cliente_titular,data_nasc_titular,genero_titular,idade_titular,id_cliente_dependente,data_nasc_dependente,genero_dependente,idade_dependente,nome_distrito,regiao,tipo_zona
12,15,1918-08-28,F,80,16,1919-02-25,M,79,Tabor,Boemia do Sul,Interior
18,24,1939-07-29,F,59,null,null,null,null,Hradec Kralove,Boemia do Leste,Interior
38,46,1940-01-30,F,58,null,null,null,null,Prachatice,Boemia do Sul,Interior
67,78,1944-06-13,F,54,null,null,null,null,Jindrichuv Hradec,Boemia do Sul,Interior
70,81,1976-04-29,M,22,82,1979-12-25,F,19,Mlada Boleslav,Boemia Central,Interior
93,110,1967-04-13,F,31,null,null,null,null,Benesov,Boemia Central,Interior
161,194,1924-12-20,F,74,null,null,null,null,Plzen-sever,Boemia do Oeste,Interior
186,226,1942-06-16,M,56,null,null,null,null,Praha,Praga,Metropolitana
190,230,1967-03-10,F,31,null,null,null,null,Ostrava-mesto,Moravia do Norte,Metropolitana
218,263,1962-05-20,M,36,null,null,null,null,Frydek-Mistek,Moravia do Norte,Interior


In [0]:
%sql
select* from portfolio.gold.dim_conta


id_conta,data,frequencia,nome_distrito,regiao,num_habitantes,salario_medio,taxa_desemprego_95,taxa_desemprego_96
576,1993-01-01,Emissao Mensal,Brno-venkov,Moravia do Sul,157042,8743,1.88,2.43
3818,1993-01-01,Emissao Mensal,Ostrava-mesto,Moravia do Norte,323870,10673,4.75,5.44
704,1993-01-01,Emissao Mensal,Brno-venkov,Moravia do Sul,157042,8743,1.88,2.43
2378,1993-01-01,Emissao Mensal,Jindrichuv Hradec,Boemia do Sul,93931,8427,1.12,1.54
2632,1993-01-02,Emissao Mensal,Karlovy Vary,Boemia do Oeste,122603,8991,1.39,2.01
1972,1993-01-02,Emissao Mensal,Vsetin,Moravia do Norte,148545,8909,4.01,5.56
1539,1993-01-03,Emissao apos Transacao,Praha,Praga,1204953,12541,0.29,0.43
793,1993-01-03,Emissao Mensal,Pardubice,Boemia do Leste,162580,9538,1.51,1.81
2484,1993-01-03,Emissao Mensal,Ostrava-mesto,Moravia do Norte,323870,10673,4.75,5.44
1695,1993-01-03,Emissao Mensal,Sumperk,Moravia do Norte,127369,8369,4.73,5.88
